# Notebook 49 - Corrected 2-state end-to-end scaffold

Notebook 48 proved that the MATLAB 2-state Kalman port works when the handoff inputs are good, but the closest current scaffold still failed because two upstream inputs are not yet MATLAB-parity:

1. Sequential Python KLT accumulates drift.
2. The full-sequence Python TimTrack alpha artifact still differs from MATLAB alpha.

This notebook corrects the scaffold by swapping those inputs one at a time, then saves a corrected downstream handoff for the final-output gate. The corrected passing handoff is deliberately labelled as a scaffold: it still uses MATLAB alpha and MATLAB aponeuroses to isolate the 2-state Kalman correction gate.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.io import loadmat

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ultrasound_tracker.geometry import line_angles_batch, normalize_angle
from ultrasound_tracker.matlab_compat import metric_row
from ultrasound_tracker.ultratimtrack_matlab_2state import (
    MatlabTwoStateKalmanConfig,
    run_matlab_2state_kalman,
)

OUT = ROOT / "results" / "notebook49_corrected_2state_end_to_end_scaffold"
OUT.mkdir(parents=True, exist_ok=True)

MATLAB_EXPORT = Path("/Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat")
MATLAB_RESULT = ROOT / "data" / "matlab" / "slow_low_2.mat"
MATLAB_TIMTRACK_EXPORT = ROOT / "results" / "notebook36_mask_parity" / "matlab_intermediate_masks_notebook36.mat"
VIDEO = ROOT / "data" / "raw" / "Test2.mp4"
NB44_NPZ = ROOT / "results" / "notebook44_ultratrack_klt_oracle_mask_gate" / "opencv_ultratrack_klt_oracle_mask_features_arrays.npz"
NB45_NPZ = ROOT / "results" / "notebook45_ultratrack_klt_one_step_affine_diagnostics" / "opencv_ultratrack_klt_one_step_affine_arrays.npz"
PY_TIMTRACK_NPZ = ROOT / "results" / "notebook38_matlab_filter_usimage_mean71" / "Test2_matlab_filter_usimage_mean71_features_arrays.npz"
NB48_SUMMARY = ROOT / "results" / "notebook48_matlab_2state_kalman_gate" / "summary.json"

paths = {
    "MATLAB numeric export": MATLAB_EXPORT,
    "MATLAB result": MATLAB_RESULT,
    "MATLAB TimTrack export": MATLAB_TIMTRACK_EXPORT,
    "video": VIDEO,
    "NB44 sequential KLT": NB44_NPZ,
    "NB45 one-step KLT": NB45_NPZ,
    "Python TimTrack alpha": PY_TIMTRACK_NPZ,
    "NB48 summary": NB48_SUMMARY,
}
for label, path in paths.items():
    print(f"{label:24s} {'OK' if path.exists() else 'MISSING'} {path}")
    if not path.exists():
        raise FileNotFoundError(path)
print("Output folder:", OUT)


MATLAB numeric export    OK /Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat
MATLAB result            OK /Users/grosbedou/PycharmProjects/NDORMS/data/matlab/slow_low_2.mat
MATLAB TimTrack export   OK /Users/grosbedou/PycharmProjects/NDORMS/results/notebook36_mask_parity/matlab_intermediate_masks_notebook36.mat
video                    OK /Users/grosbedou/PycharmProjects/NDORMS/data/raw/Test2.mp4
NB44 sequential KLT      OK /Users/grosbedou/PycharmProjects/NDORMS/results/notebook44_ultratrack_klt_oracle_mask_gate/opencv_ultratrack_klt_oracle_mask_features_arrays.npz
NB45 one-step KLT        OK /Users/grosbedou/PycharmProjects/NDORMS/results/notebook45_ultratrack_klt_one_step_affine_diagnostics/opencv_ultratrack_klt_one_step_affine_arrays.npz
Python TimTrack alpha    OK /Users/grosbedou/PycharmProjects/NDORMS/results/notebook38_matlab_filter_usimage_mean71/Test2_matlab_filter_usimage_mean71_features_arrays.npz
NB48 summary             OK /Users/grosbedou/PycharmProj

## Load references and handoff inputs

The reference for this correction is the same `UTT_numeric_export.mat` used in notebook 48. That keeps the Kalman correction gate comparable with the previous notebook.

Coordinate convention used below:

- MATLAB fascicle endpoint fields are stored as `[deep, superficial]` pairs.
- Python segments are `[x_superficial, y_superficial, x_deep, y_deep]`.
- MATLAB aponeurosis endpoint fields are already left-to-right lines and are kept as `[x1, y1, x2, y2]`.


In [2]:
mat_root = loadmat(MATLAB_EXPORT, simplify_cells=True)["UTT_numeric_export"]
region = mat_root["Region"]
fascicle = region["Fascicle"]
geofeatures = list(np.asarray(mat_root["geofeatures"], dtype=object).reshape(-1))

height = int(mat_root["vidHeight"])
width = int(mat_root["vidWidth"])
mat_n = int(mat_root["NumFrames"])
mm_per_px = float(mat_root["ID"]) / height


def matlab_pair_series(values: object, n: int | None = None) -> np.ndarray:
    out = []
    for value in np.asarray(values, dtype=object).reshape(-1):
        arr = np.asarray(value, dtype=float).reshape(-1)
        out.append(arr[:2] if arr.size >= 2 else [np.nan, np.nan])
    result = np.asarray(out, dtype=float)
    return result if n is None else result[:n]


def matlab_fascicle_segments(x_field: str, y_field: str, n: int | None = None) -> np.ndarray:
    x = matlab_pair_series(fascicle[x_field], n=n)
    y = matlab_pair_series(fascicle[y_field], n=n)
    return np.column_stack([x[:, 1], y[:, 1], x[:, 0], y[:, 0]])


def matlab_apo_segments(prefix: str, n: int | None = None) -> np.ndarray:
    x = matlab_pair_series(region[f"{prefix}_x"], n=n)
    y = matlab_pair_series(region[f"{prefix}_y"], n=n)
    return np.column_stack([x[:, 0], y[:, 0], x[:, 1], y[:, 1]])


def matlab_state_series(field: str, n: int | None = None) -> np.ndarray:
    return matlab_pair_series(fascicle[field], n=n)


def normalized_angles_deg(segments: np.ndarray) -> np.ndarray:
    return np.asarray(
        [normalize_angle(v, degrees=True) for v in line_angles_batch(segments, degrees=True)],
        dtype=float,
    )

mat_raw_klt = matlab_fascicle_segments("fas_x_original", "fas_y_original", n=mat_n)
mat_sup = matlab_apo_segments("sup", n=mat_n)
mat_deep = matlab_apo_segments("deep", n=mat_n)
mat_timtrack_alpha = np.asarray([float(entry["alpha"]) for entry in geofeatures], dtype=float)[:mat_n]

mat_final = {
    "FL_mm": np.asarray(region["fas_length"], dtype=float).reshape(-1)[:mat_n],
    "PEN_deg": np.asarray(region["fas_pen"], dtype=float).reshape(-1)[:mat_n],
    "ANG_deg": np.asarray(region["fas_ang"], dtype=float).reshape(-1)[:mat_n],
    "X_plus": matlab_state_series("X_plus", n=mat_n),
    "X_minus": matlab_state_series("X_minus", n=mat_n),
}

nb44 = np.load(NB44_NPZ, allow_pickle=True)
nb45 = np.load(NB45_NPZ, allow_pickle=True)
py_timtrack = np.load(PY_TIMTRACK_NPZ, allow_pickle=True)

py_seq_klt = np.asarray(nb44["fascicle_segments"], dtype=float)[:mat_n]
py_one_step_klt = np.asarray(nb45["fascicle_segments"], dtype=float)[:mat_n]
py_timtrack_alpha = np.asarray(py_timtrack["ANG_deg"], dtype=float).reshape(-1)[:mat_n]

config = MatlabTwoStateKalmanConfig(
    q_parameter=float(mat_root["Q"]),
    x_measurement_variance=float(mat_root["X"]),
    alpha_measurement_variance=float(np.asarray(mat_root["R"], dtype=float).reshape(-1)[0]),
    n_start_frames=int(mat_root["NS"]),
    run_smoother=True,
)

print({
    "frames": mat_n,
    "image_size": (height, width),
    "mm_per_px": mm_per_px,
    "config": config,
    "mat_raw_klt_shape": mat_raw_klt.shape,
    "py_seq_klt_shape": py_seq_klt.shape,
    "py_one_step_klt_shape": py_one_step_klt.shape,
    "py_alpha_shape": py_timtrack_alpha.shape,
})


{'frames': 2666, 'image_size': (562, 706), 'mm_per_px': 0.09021352313167261, 'config': MatlabTwoStateKalmanConfig(q_parameter=0.01, x_measurement_variance=100.0, alpha_measurement_variance=3.055292111054342, n_start_frames=1, run_smoother=True), 'mat_raw_klt_shape': (2666, 4), 'py_seq_klt_shape': (2666, 4), 'py_one_step_klt_shape': (2666, 4), 'py_alpha_shape': (2666,)}


## Direct alpha and KLT blocker checks

Before correcting the scaffold, check whether the Python alpha problem is a simple sign or 180-degree convention issue. It is not: the current full-sequence Python alpha has a real signal mismatch relative to the MATLAB alpha used by the downstream Kalman stage.


In [3]:
def metric(label: str, reference, estimate, unit: str, target_rmse: float | None = None) -> dict:
    row = metric_row(label, reference, estimate)
    row["unit"] = unit
    row["target_rmse"] = target_rmse
    row["passes"] = bool(target_rmse is None or row["rmse"] <= target_rmse)
    return row


alpha_candidates = {
    "python_alpha": py_timtrack_alpha,
    "python_alpha_plus_180": py_timtrack_alpha + 180.0,
    "negative_python_alpha": -py_timtrack_alpha,
    "abs_python_alpha": np.abs(py_timtrack_alpha),
}
alpha_rows = [metric(name, mat_timtrack_alpha, values, "deg", target_rmse=1.0) for name, values in alpha_candidates.items()]

for offset in [-3, -2, -1, 1, 2, 3]:
    if offset < 0:
        ref = mat_timtrack_alpha[-offset:]
        est = py_timtrack_alpha[: len(ref)]
    else:
        ref = mat_timtrack_alpha[:-offset]
        est = py_timtrack_alpha[offset:]
    alpha_rows.append(metric(f"python_alpha_offset_{offset:+d}", ref, est, "deg", target_rmse=1.0))

alpha_diag = pd.DataFrame(alpha_rows)
alpha_diag = alpha_diag[["comparison", "unit", "n", "bias", "mae", "rmse", "corr", "target_rmse", "passes"]]
alpha_diag_path = OUT / "alpha_source_diagnostics.csv"
alpha_diag.to_csv(alpha_diag_path, index=False)
print("Saved:", alpha_diag_path)
display(alpha_diag.round(4))

klt_diag_rows = [
    metric("sequential_klt_angle", normalized_angles_deg(mat_raw_klt), normalized_angles_deg(py_seq_klt), "deg", 1.0),
    metric("one_step_klt_angle", normalized_angles_deg(mat_raw_klt), normalized_angles_deg(py_one_step_klt), "deg", 1.0),
    metric("sequential_klt_x_sup", mat_raw_klt[:, 0], py_seq_klt[:, 0], "px", 2.0),
    metric("one_step_klt_x_sup", mat_raw_klt[:, 0], py_one_step_klt[:, 0], "px", 2.0),
]
klt_diag = pd.DataFrame(klt_diag_rows)
klt_diag_path = OUT / "klt_source_diagnostics.csv"
klt_diag.to_csv(klt_diag_path, index=False)
print("Saved:", klt_diag_path)
display(klt_diag.round(4))


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook49_corrected_2state_end_to_end_scaffold/alpha_source_diagnostics.csv


,comparison,unit,n,bias,mae,rmse,corr,target_rmse,passes
0,python_alpha,deg,2666,-4.3453,4.6735,6.2015,0.5685,1.0,False
1,python_alpha_plus_180,deg,2666,175.6547,175.6547,175.7104,0.5685,1.0,False
2,negative_python_alpha,deg,2666,-46.7706,46.7706,47.4381,-0.5685,1.0,False
3,abs_python_alpha,deg,2666,-4.3453,4.6735,6.2015,0.5685,1.0,False
4,python_alpha_offset_-3,deg,2663,-4.3513,4.6908,6.2340,0.5589,1.0,False
5,python_alpha_offset_-2,deg,2664,-4.3493,4.6815,6.2209,0.5629,1.0,False
6,python_alpha_offset_-1,deg,2665,-4.3473,4.6846,6.2281,0.5601,1.0,False
7,python_alpha_offset_+1,deg,2665,-4.3462,4.6775,6.2157,0.5640,1.0,False
8,python_alpha_offset_+2,deg,2664,-4.3474,4.6744,6.2173,0.5638,1.0,False
9,python_alpha_offset_+3,deg,2663,-4.3489,4.6827,6.2109,0.5662,1.0,False


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook49_corrected_2state_end_to_end_scaffold/klt_source_diagnostics.csv


,comparison,n,bias,mae,rmse,corr,unit,target_rmse,passes
0,sequential_klt_angle,2666,-5.0708,5.0715,6.1182,0.7095,deg,1.0,False
1,one_step_klt_angle,2666,0.0016,0.1670,0.3013,0.9981,deg,1.0,True
2,sequential_klt_x_sup,2666,8.0301,13.4554,15.8097,0.0898,px,2.0,False
3,one_step_klt_x_sup,2666,-0.0068,0.6555,1.2424,0.9950,px,2.0,True


## Correct the scaffold by swapping inputs one at a time

The variants are intentionally mechanical:

- `current_scaffold`: current sequential Python KLT plus current full-sequence Python alpha.
- `klt_corrected_only`: notebook 45 local one-step KLT plus current full-sequence Python alpha.
- `alpha_corrected_only`: current sequential Python KLT plus MATLAB alpha.
- `corrected_handoff`: notebook 45 local one-step KLT plus MATLAB alpha.
- `oracle_matlab_inputs`: MATLAB raw KLT plus MATLAB alpha.

All variants still use MATLAB aponeuroses. That isolates the fascicle KLT/alpha/Kalman correction from the separate aponeurosis Kalman port.


In [4]:
variants = {
    "current_scaffold": {
        "klt_segments": py_seq_klt,
        "alpha": py_timtrack_alpha,
        "description": "sequential Python KLT + current Python TimTrack alpha + MATLAB aponeuroses",
        "uses_reference_alpha": False,
        "uses_local_klt": False,
    },
    "klt_corrected_only": {
        "klt_segments": py_one_step_klt,
        "alpha": py_timtrack_alpha,
        "description": "notebook45 local one-step Python KLT + current Python TimTrack alpha + MATLAB aponeuroses",
        "uses_reference_alpha": False,
        "uses_local_klt": True,
    },
    "alpha_corrected_only": {
        "klt_segments": py_seq_klt,
        "alpha": mat_timtrack_alpha,
        "description": "sequential Python KLT + MATLAB TimTrack alpha + MATLAB aponeuroses",
        "uses_reference_alpha": True,
        "uses_local_klt": False,
    },
    "corrected_handoff": {
        "klt_segments": py_one_step_klt,
        "alpha": mat_timtrack_alpha,
        "description": "notebook45 local one-step Python KLT + MATLAB TimTrack alpha + MATLAB aponeuroses",
        "uses_reference_alpha": True,
        "uses_local_klt": True,
    },
    "oracle_matlab_inputs": {
        "klt_segments": mat_raw_klt,
        "alpha": mat_timtrack_alpha,
        "description": "MATLAB raw KLT + MATLAB TimTrack alpha + MATLAB aponeuroses",
        "uses_reference_alpha": True,
        "uses_local_klt": False,
    },
}

results = {}
for name, spec in variants.items():
    results[name] = run_matlab_2state_kalman(
        spec["klt_segments"],
        spec["alpha"],
        mat_sup,
        mat_deep,
        config=config,
        mm_per_pixel=mm_per_px,
    )

variant_npz = OUT / "corrected_2state_variant_arrays.npz"
np.savez(
    variant_npz,
    frame=np.arange(mat_n, dtype=np.int32),
    matlab_FL_mm=mat_final["FL_mm"],
    matlab_PEN_deg=mat_final["PEN_deg"],
    matlab_ANG_deg=mat_final["ANG_deg"],
    matlab_alpha_deg=mat_timtrack_alpha,
    python_alpha_deg=py_timtrack_alpha,
    **{f"{name}_FL_mm": result["FL_mm"] for name, result in results.items()},
    **{f"{name}_PEN_deg": result["PEN_deg"] for name, result in results.items()},
    **{f"{name}_ANG_deg": result["ANG_deg"] for name, result in results.items()},
)
print("Saved:", variant_npz)


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook49_corrected_2state_end_to_end_scaffold/corrected_2state_variant_arrays.npz


## Full-sequence final-output gate

Targets:

- `FL` RMSE < 2 mm
- `PEN` RMSE < 1 degree
- `ANG` RMSE < 1 degree


In [5]:
metric_rows = []
for name, result in results.items():
    for comparison, unit, target in [
        ("FL_mm", "mm", 2.0),
        ("PEN_deg", "deg", 1.0),
        ("ANG_deg", "deg", 1.0),
    ]:
        row = metric(
            f"{name}_{comparison}",
            mat_final[comparison],
            result[comparison],
            unit,
            target_rmse=target,
        )
        row["variant"] = name
        row["description"] = variants[name]["description"]
        metric_rows.append(row)

metrics = pd.DataFrame(metric_rows)
metrics = metrics[["variant", "description", "comparison", "unit", "n", "bias", "mae", "rmse", "corr", "target_rmse", "passes"]]
metrics_path = OUT / "corrected_2state_full_gate_metrics.csv"
metrics.to_csv(metrics_path, index=False)
print("Saved:", metrics_path)
display(metrics.round(4))

summary_rows = []
for name in variants:
    group = metrics[metrics["variant"] == name]
    summary_rows.append({
        "variant": name,
        "description": variants[name]["description"],
        "uses_local_klt": variants[name]["uses_local_klt"],
        "uses_reference_alpha": variants[name]["uses_reference_alpha"],
        "FL_rmse_mm": float(group.loc[group["comparison"] == f"{name}_FL_mm", "rmse"].iloc[0]),
        "PEN_rmse_deg": float(group.loc[group["comparison"] == f"{name}_PEN_deg", "rmse"].iloc[0]),
        "ANG_rmse_deg": float(group.loc[group["comparison"] == f"{name}_ANG_deg", "rmse"].iloc[0]),
        "passes_final_gate": bool(group["passes"].all()),
    })
summary_df = pd.DataFrame(summary_rows)
summary_path = OUT / "corrected_2state_variant_summary.csv"
summary_df.to_csv(summary_path, index=False)
print("Saved:", summary_path)
display(summary_df.round(4))


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook49_corrected_2state_end_to_end_scaffold/corrected_2state_full_gate_metrics.csv


,variant,description,comparison,unit,n,bias,mae,rmse,corr,target_rmse,passes
0,current_scaffold,sequential Python KLT + current Python TimTrac...,current_scaffold_FL_mm,mm,2666,8.9539,8.9563,11.2085,0.7010,2.0,False
1,current_scaffold,sequential Python KLT + current Python TimTrac...,current_scaffold_PEN_deg,deg,2666,-4.3265,4.3272,5.4290,0.6539,1.0,False
2,current_scaffold,sequential Python KLT + current Python TimTrac...,current_scaffold_ANG_deg,deg,2666,-4.3265,4.3272,5.4290,0.7009,1.0,False
3,klt_corrected_only,notebook45 local one-step Python KLT + current...,klt_corrected_only_FL_mm,mm,2666,9.7084,9.7086,10.3271,0.9738,2.0,False
4,klt_corrected_only,notebook45 local one-step Python KLT + current...,klt_corrected_only_PEN_deg,deg,2666,-4.1866,4.1867,4.3000,0.9770,1.0,False
5,klt_corrected_only,notebook45 local one-step Python KLT + current...,klt_corrected_only_ANG_deg,deg,2666,-4.1866,4.1867,4.3000,0.9794,1.0,False
6,alpha_corrected_only,sequential Python KLT + MATLAB TimTrack alpha ...,alpha_corrected_only_FL_mm,mm,2666,-0.4822,5.7090,6.8688,0.6860,2.0,False
7,alpha_corrected_only,sequential Python KLT + MATLAB TimTrack alpha ...,alpha_corrected_only_PEN_deg,deg,2666,-0.1573,2.7955,3.2928,0.6574,1.0,False
8,alpha_corrected_only,sequential Python KLT + MATLAB TimTrack alpha ...,alpha_corrected_only_ANG_deg,deg,2666,-0.1573,2.7955,3.2928,0.6982,1.0,False
9,corrected_handoff,notebook45 local one-step Python KLT + MATLAB ...,corrected_handoff_FL_mm,mm,2666,-0.0248,0.5666,0.7530,0.9968,2.0,True


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook49_corrected_2state_end_to_end_scaffold/corrected_2state_variant_summary.csv


,variant,description,uses_local_klt,uses_reference_alpha,FL_rmse_mm,PEN_rmse_deg,ANG_rmse_deg,passes_final_gate
0,current_scaffold,sequential Python KLT + current Python TimTrac...,False,False,11.2085,5.4290,5.4290,False
1,klt_corrected_only,notebook45 local one-step Python KLT + current...,True,False,10.3271,4.3000,4.3000,False
2,alpha_corrected_only,sequential Python KLT + MATLAB TimTrack alpha ...,False,True,6.8688,3.2928,3.2928,False
3,corrected_handoff,notebook45 local one-step Python KLT + MATLAB ...,True,True,0.7530,0.4097,0.4097,True
4,oracle_matlab_inputs,MATLAB raw KLT + MATLAB TimTrack alpha + MATLA...,False,True,0.5603,0.2657,0.2657,True


## Nine-frame validation slice

This checks the same focus frames used in the TimTrack notebooks. The full-sequence gate above is the actual pass/fail gate; this slice makes it easy to compare against the earlier 9-frame work.


In [6]:
focus_frames = np.asarray([0, 122, 533, 691, 955, 1066, 1600, 2133, 2665], dtype=int)
focus_rows = []
for name, result in results.items():
    for comparison, unit, target in [
        ("FL_mm", "mm", 2.0),
        ("PEN_deg", "deg", 1.0),
        ("ANG_deg", "deg", 1.0),
    ]:
        row = metric(
            f"{name}_{comparison}",
            mat_final[comparison][focus_frames],
            result[comparison][focus_frames],
            unit,
            target_rmse=target,
        )
        row["variant"] = name
        focus_rows.append(row)

focus_metrics = pd.DataFrame(focus_rows)
focus_metrics = focus_metrics[["variant", "comparison", "unit", "n", "bias", "mae", "rmse", "corr", "target_rmse", "passes"]]
focus_path = OUT / "corrected_2state_focus9_metrics.csv"
focus_metrics.to_csv(focus_path, index=False)
print("Saved:", focus_path)
display(focus_metrics.round(4))


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook49_corrected_2state_end_to_end_scaffold/corrected_2state_focus9_metrics.csv


,variant,comparison,unit,n,bias,mae,rmse,corr,target_rmse,passes
0,current_scaffold,current_scaffold_FL_mm,mm,9,3.3678,3.3678,4.7930,0.9950,2.0,False
1,current_scaffold,current_scaffold_PEN_deg,deg,9,-1.9631,1.9631,3.0134,0.9907,1.0,False
2,current_scaffold,current_scaffold_ANG_deg,deg,9,-1.9631,1.9631,3.0134,0.9936,1.0,False
3,klt_corrected_only,klt_corrected_only_FL_mm,mm,9,10.6507,10.6507,11.8428,0.9203,2.0,False
4,klt_corrected_only,klt_corrected_only_PEN_deg,deg,9,-3.9741,3.9741,4.3312,0.9391,1.0,False
5,klt_corrected_only,klt_corrected_only_ANG_deg,deg,9,-3.9741,3.9741,4.3312,0.9500,1.0,False
6,alpha_corrected_only,alpha_corrected_only_FL_mm,mm,9,-4.8259,6.0603,7.4093,0.8753,2.0,False
7,alpha_corrected_only,alpha_corrected_only_PEN_deg,deg,9,1.7819,2.6728,3.1069,0.8645,1.0,False
8,alpha_corrected_only,alpha_corrected_only_ANG_deg,deg,9,1.7819,2.6728,3.1069,0.8969,1.0,False
9,corrected_handoff,corrected_handoff_FL_mm,mm,9,0.5003,0.9294,1.2242,0.9959,2.0,True


## Save corrected handoff and run the shared parity script

`corrected_handoff` is the corrected downstream scaffold for the 2-state Kalman gate. It is intentionally not claimed as the complete Python pipeline because it still uses MATLAB alpha and MATLAB aponeuroses.

This cell saves a small script-compatible NPZ and runs `scripts/compare_ultratimtrack_parity.py` to generate `parity_metrics.csv` for the final rows.


In [7]:
corrected = results["corrected_handoff"]
corrected_npz = OUT / "corrected_handoff_final_outputs_arrays.npz"
np.savez(
    corrected_npz,
    frame=np.arange(mat_n, dtype=np.int32),
    FL_mm=corrected["FL_mm"],
    PEN_deg=corrected["PEN_deg"],
    ANG_deg=corrected["ANG_deg"],
    fascicle_segments=corrected["fascicle_segments"],
    fascicle_end_segments=corrected["fascicle_end_segments"],
    X_plus=corrected["X_plus"],
    X_minus=corrected["X_minus"],
)

corrected_csv = OUT / "corrected_handoff_final_outputs.csv"
pd.DataFrame({
    "frame": np.arange(mat_n, dtype=np.int32),
    "FL_mm": corrected["FL_mm"],
    "PEN_deg": corrected["PEN_deg"],
    "ANG_deg": corrected["ANG_deg"],
}).to_csv(corrected_csv, index=False)
print("Saved:", corrected_npz)
print("Saved:", corrected_csv)

PARITY_CSV = OUT / "parity_metrics.csv"
cmd = [
    sys.executable,
    str(ROOT / "scripts" / "compare_ultratimtrack_parity.py"),
    "--matlab-result", str(MATLAB_RESULT),
    "--python-utt", str(corrected_npz),
    "--python-timtrack", str(PY_TIMTRACK_NPZ),
    "--video", str(VIDEO),
    "--utt-export", str(MATLAB_EXPORT),
    "--matlab-timtrack-export", str(MATLAB_TIMTRACK_EXPORT),
    "--out-csv", str(PARITY_CSV),
]
print(" ".join(cmd))
run = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
print(run.stdout)
if run.stderr:
    print(run.stderr)
if run.returncode != 0:
    raise RuntimeError(f"compare_ultratimtrack_parity.py failed with exit code {run.returncode}")

parity_df = pd.read_csv(PARITY_CSV)
print("Saved:", PARITY_CSV)
display(parity_df.round(4))

final_targets = {"final_FL_mm": 2.0, "final_PEN_deg": 1.0, "final_ANG_deg": 1.0}
final_script_gate = parity_df[parity_df["comparison"].isin(final_targets)].copy()
final_script_gate["target_rmse"] = final_script_gate["comparison"].map(final_targets)
final_script_gate["passes"] = final_script_gate["rmse"] <= final_script_gate["target_rmse"]
display(final_script_gate.round(4))


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook49_corrected_2state_end_to_end_scaffold/corrected_handoff_final_outputs_arrays.npz
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook49_corrected_2state_end_to_end_scaffold/corrected_handoff_final_outputs.csv
/Users/grosbedou/PycharmProjects/NDORMS/.venv/bin/python /Users/grosbedou/PycharmProjects/NDORMS/scripts/compare_ultratimtrack_parity.py --matlab-result /Users/grosbedou/PycharmProjects/NDORMS/data/matlab/slow_low_2.mat --python-utt /Users/grosbedou/PycharmProjects/NDORMS/results/notebook49_corrected_2state_end_to_end_scaffold/corrected_handoff_final_outputs_arrays.npz --python-timtrack /Users/grosbedou/PycharmProjects/NDORMS/results/notebook38_matlab_filter_usimage_mean71/Test2_matlab_filter_usimage_mean71_features_arrays.npz --video /Users/grosbedou/PycharmProjects/NDORMS/data/raw/Test2.mp4 --utt-export /Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat --matlab-timtrack-export /Users

MATLAB result: /Users/grosbedou/PycharmProjects/NDORMS/data/matlab/slow_low_2.mat
MATLAB TimTrack export: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook36_mask_parity/matlab_intermediate_masks_notebook36.mat
Python final:  /Users/grosbedou/PycharmProjects/NDORMS/results/notebook49_corrected_2state_end_to_end_scaffold/corrected_handoff_final_outputs_arrays.npz
Python TimTrack-like: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook38_matlab_filter_usimage_mean71/Test2_matlab_filter_usimage_mean71_features_arrays.npz
image_depth_mm=50.7
image_height_px=562
image_width_px=706
mm_per_pixel=0.09021352
apox_1b=[20.0, 94.0, 168.0, 242.0, 316.0, 390.0, 464.0, 538.0, 612.0, 686.0]

comparison                                n       bias        mae       rmse     corr
-------------------------------------------------------------------------------------
final_FL_mm                            2666    -0.0248     0.5666     0.7530   0.9968
final_PEN_deg                          2666 

,comparison,n,bias,mae,rmse,corr
0,final_FL_mm,2666,-0.0248,0.5666,0.7530,0.9968
1,final_PEN_deg,2666,0.0254,0.2974,0.4097,0.9957
2,final_ANG_deg,2666,0.0254,0.2974,0.4097,0.9961
3,timtrack_alpha_deg,9,0.0000,0.2222,0.4082,0.9854
4,timtrack_phi_vs_python_pen_deg,9,-0.0229,0.2295,0.3935,0.9888
5,timtrack_formula_faslen_px,9,2.1482,11.3478,18.4763,0.9851
6,timtrack_gamma_deep_apo_deg,9,0.0823,0.0823,0.1380,0.9673
7,timtrack_betha_super_apo_deg,9,0.0229,0.0500,0.0831,0.9670
8,timtrack_super_pos_y1,9,1.2743,1.2743,1.4441,0.9688
9,timtrack_super_pos_y2,9,0.9920,0.9920,1.0421,0.9674


,comparison,n,bias,mae,rmse,corr,target_rmse,passes
0,final_FL_mm,2666,-0.0248,0.5666,0.7530,0.9968,2.0,True
1,final_PEN_deg,2666,0.0254,0.2974,0.4097,0.9957,1.0,True
2,final_ANG_deg,2666,0.0254,0.2974,0.4097,0.9961,1.0,True


## Readiness decision

This correction makes the downstream 2-state gate pass with the best currently validated handoff, but it also narrows the remaining full-Python blockers:

- Correcting KLT alone is not enough while full-sequence Python alpha is still wrong.
- Correcting alpha alone is not enough while sequential KLT drift remains.
- Correcting both KLT handoff and alpha makes the 2-state final gate pass.

So the MATLAB 2-state Kalman gate is ready as a separate downstream correction gate. The complete Python end-to-end pipeline is not ready for the final gate until full-sequence TimTrack alpha and sequential KLT propagation are corrected or replaced by an equivalent validated handoff.


In [8]:
current_pass = bool(summary_df.loc[summary_df["variant"] == "current_scaffold", "passes_final_gate"].iloc[0])
klt_only_pass = bool(summary_df.loc[summary_df["variant"] == "klt_corrected_only", "passes_final_gate"].iloc[0])
alpha_only_pass = bool(summary_df.loc[summary_df["variant"] == "alpha_corrected_only", "passes_final_gate"].iloc[0])
corrected_pass = bool(summary_df.loc[summary_df["variant"] == "corrected_handoff", "passes_final_gate"].iloc[0])
oracle_pass = bool(summary_df.loc[summary_df["variant"] == "oracle_matlab_inputs", "passes_final_gate"].iloc[0])
script_final_pass = bool(final_script_gate["passes"].all())

readiness = pd.DataFrame([
    {
        "gate": "MATLAB 2-state Kalman port",
        "status": "passes oracle and corrected-handoff final gates" if oracle_pass and corrected_pass else "not passing",
        "ready_for_next": bool(oracle_pass and corrected_pass),
    },
    {
        "gate": "Corrected downstream handoff",
        "status": "passes final FL/PEN/ANG targets, but uses MATLAB alpha and aponeuroses",
        "ready_for_next": corrected_pass,
    },
    {
        "gate": "Current Python end-to-end scaffold",
        "status": "passes" if current_pass else "not ready; sequential KLT plus full Python alpha fails final gate",
        "ready_for_next": current_pass,
    },
    {
        "gate": "KLT correction alone",
        "status": "passes" if klt_only_pass else "not enough while full Python alpha remains off",
        "ready_for_next": klt_only_pass,
    },
    {
        "gate": "Alpha correction alone",
        "status": "passes" if alpha_only_pass else "not enough while sequential KLT drift remains",
        "ready_for_next": alpha_only_pass,
    },
    {
        "gate": "Shared parity script on corrected handoff",
        "status": "final rows pass" if script_final_pass else "final rows fail",
        "ready_for_next": script_final_pass,
    },
])
readiness_path = OUT / "readiness.csv"
readiness.to_csv(readiness_path, index=False)
print("Saved:", readiness_path)
display(readiness)

summary = {
    "notebook": "49_corrected_2state_end_to_end_scaffold.ipynb",
    "module": "ultrasound_tracker.ultratimtrack_matlab_2state",
    "current_end_to_end_scaffold_passes": current_pass,
    "klt_corrected_only_passes": klt_only_pass,
    "alpha_corrected_only_passes": alpha_only_pass,
    "corrected_handoff_passes": corrected_pass,
    "oracle_matlab_inputs_passes": oracle_pass,
    "script_final_rows_pass": script_final_pass,
    "current_scaffold_FL_rmse_mm": float(summary_df.loc[summary_df["variant"] == "current_scaffold", "FL_rmse_mm"].iloc[0]),
    "klt_corrected_only_FL_rmse_mm": float(summary_df.loc[summary_df["variant"] == "klt_corrected_only", "FL_rmse_mm"].iloc[0]),
    "alpha_corrected_only_FL_rmse_mm": float(summary_df.loc[summary_df["variant"] == "alpha_corrected_only", "FL_rmse_mm"].iloc[0]),
    "corrected_handoff_FL_rmse_mm": float(summary_df.loc[summary_df["variant"] == "corrected_handoff", "FL_rmse_mm"].iloc[0]),
    "corrected_handoff_PEN_rmse_deg": float(summary_df.loc[summary_df["variant"] == "corrected_handoff", "PEN_rmse_deg"].iloc[0]),
    "corrected_handoff_ANG_rmse_deg": float(summary_df.loc[summary_df["variant"] == "corrected_handoff", "ANG_rmse_deg"].iloc[0]),
    "python_alpha_rmse_deg": float(alpha_diag.loc[alpha_diag["comparison"] == "python_alpha", "rmse"].iloc[0]),
    "remaining_blockers": [
        "full-sequence Python TimTrack alpha parity",
        "sequential KLT propagation drift",
        "aponeurosis state estimator port for complete Python final output",
    ],
}
summary_path = OUT / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print("Saved:", summary_path)
display(Markdown(
    "**Decision:** corrected downstream handoff passes the final gate, "
    "but the complete Python end-to-end pipeline is still blocked by full-sequence alpha and sequential KLT."
))
print(json.dumps(summary, indent=2))


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook49_corrected_2state_end_to_end_scaffold/readiness.csv


,gate,status,ready_for_next
0,MATLAB 2-state Kalman port,passes oracle and corrected-handoff final gates,True
1,Corrected downstream handoff,"passes final FL/PEN/ANG targets, but uses MATL...",True
2,Current Python end-to-end scaffold,not ready; sequential KLT plus full Python alp...,False
3,KLT correction alone,not enough while full Python alpha remains off,False
4,Alpha correction alone,not enough while sequential KLT drift remains,False
5,Shared parity script on corrected handoff,final rows pass,True


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook49_corrected_2state_end_to_end_scaffold/summary.json


**Decision:** corrected downstream handoff passes the final gate, but the complete Python end-to-end pipeline is still blocked by full-sequence alpha and sequential KLT.

{
  "notebook": "49_corrected_2state_end_to_end_scaffold.ipynb",
  "module": "ultrasound_tracker.ultratimtrack_matlab_2state",
  "current_end_to_end_scaffold_passes": false,
  "klt_corrected_only_passes": false,
  "alpha_corrected_only_passes": false,
  "corrected_handoff_passes": true,
  "oracle_matlab_inputs_passes": true,
  "script_final_rows_pass": true,
  "current_scaffold_FL_rmse_mm": 11.208491029219624,
  "klt_corrected_only_FL_rmse_mm": 10.327051272212927,
  "alpha_corrected_only_FL_rmse_mm": 6.868819502554928,
  "corrected_handoff_FL_rmse_mm": 0.7529551324064312,
  "corrected_handoff_PEN_rmse_deg": 0.409723832187515,
  "corrected_handoff_ANG_rmse_deg": 0.40972383218751485,
  "python_alpha_rmse_deg": 6.201488399133804,
  "remaining_blockers": [
    "full-sequence Python TimTrack alpha parity",
    "sequential KLT propagation drift",
    "aponeurosis state estimator port for complete Python final output"
  ]
}
